In [ ]:
# ==============================================================================
# 1. LIBRARY IMPORTS
# ==============================================================================
import re
import warnings
import numpy as np
import pandas as pd
import optuna

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)


# ==============================================================================
# 2. CUSTOM ALGORITHM DEFINITION
# ==============================================================================
def softmax(F):
    """Compute row-wise softmax."""
    expF = np.exp(F - np.max(F, axis=1, keepdims=True))
    return expF / np.sum(expF, axis=1, keepdims=True)

class CustomMulticlassGradientBoostingClassifier:
    """
    Custom multiclass Gradient Boosting classifier with LDA.
    """
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.estimators = []
        self.lda_transforms = []
        self.initial_logit = None
        self.classes_ = None

    def fit(self, X, y):
        # Convert to numpy array if it's a pandas DataFrame
        if isinstance(X, pd.DataFrame):
            X = X.to_numpy()

        n_samples = X.shape[0]
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        one_hot_y = np.eye(n_classes)[np.searchsorted(self.classes_, y)]
        
        prior = np.mean(one_hot_y, axis=0)
        prior = np.clip(prior, 1e-5, 1 - 1e-5)
        self.initial_logit = np.log(prior)
        
        F = np.tile(self.initial_logit, (n_samples, 1))
        
        for m in range(self.n_estimators):
            if m == 0:
                lda = LinearDiscriminantAnalysis(n_components=None)
                X_lda = lda.fit_transform(X, y)
            else:
                p = softmax(F)
                residuals = one_hot_y - p
                labels = np.argmax(residuals, axis=1)
                lda = LinearDiscriminantAnalysis(n_components=None)
                X_lda = lda.fit_transform(X, labels)
            
            self.lda_transforms.append(lda)
            
            p = softmax(F)
            residuals = one_hot_y - p
            
            estimators_m = []
            for k in range(n_classes):
                reg = DecisionTreeRegressor(max_depth=self.max_depth, random_state=42)
                reg.fit(X_lda, residuals[:, k])
                estimators_m.append(reg)
                update = reg.predict(X_lda)
                F[:, k] += self.learning_rate * update
            
            self.estimators.append(estimators_m)
        return self

    def predict_proba(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.to_numpy()

        n_samples = X.shape[0]
        F = np.tile(self.initial_logit, (n_samples, 1))
        for lda, estimators_m in zip(self.lda_transforms, self.estimators):
            X_lda = lda.transform(X)
            for k, reg in enumerate(estimators_m):
                F[:, k] += self.learning_rate * reg.predict(X_lda)
        return softmax(F)

    def predict(self, X):
        proba = self.predict_proba(X)
        class_indices = np.argmax(proba, axis=1)
        return self.classes_[class_indices]
        
    def get_params(self, deep=True):
        """ Required for scikit-learn compatibility (e.g., cross_val_score). """
        return {
            'n_estimators': self.n_estimators,
            'learning_rate': self.learning_rate,
            'max_depth': self.max_depth
        }


# ==============================================================================
# 3. DATA LOADING AND PREPROCESSING
# ==============================================================================
print("Start: Loading and preparing data...")

# Load data
try:
    original_train = pd.read_csv("Data/train.csv")
    # Provide column names to the test set during loading
    original_test = pd.read_csv("Data/test.csv", names=list(original_train.columns), header=0)
except FileNotFoundError:
    print("Error: 'train.csv' or 'test.csv' not found. Make sure they are in the same directory.")
    exit()

train_df = original_train.copy()
test_df = original_test.copy()

# Clean column names for compatibility
def clean_col_names(df):
    cols = df.columns
    new_cols = [re.sub(r"[^A-Za-z0-9_]+", "_", col) for col in cols]
    df.columns = new_cols
    return df

train_df = clean_col_names(train_df)
test_df = clean_col_names(test_df)

# Separate features (X) and target (y)
X = train_df.drop(columns="Activity")
y = train_df["Activity"]
X_test = test_df.drop(columns="Activity")
y_test = test_df["Activity"]

# Encode the target into numeric values
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_encoded_test = label_encoder.transform(y_test)  # Use transform, not fit_transform, on the test set

print("Data loaded and ready.")
print(f"Training set shape: {X.shape}")
print(f"Test set shape: {X_test.shape}")
print("-" * 50)


# ==============================================================================
# 4. TUNING PIPELINE WITH OPTUNA
# ==============================================================================
def objective(trial):
    """ Objective function for Optuna to maximize. """
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 2000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 8),
    }

    model = CustomMulticlassGradientBoostingClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X, y_encoded, n_jobs=-1, cv=cv, scoring='accuracy')
    return score.mean()

print("Starting hyperparameter optimization with Optuna...")
study = optuna.create_study(direction='maximize')
# Increase n_trials for a more thorough search (e.g., 100)
study.optimize(objective, n_trials=15, show_progress_bar=True)

# Optimization results
best_params = study.best_params
best_accuracy_cv = study.best_value

print("\n" + "="*50)
print("TUNING COMPLETED")
print("="*50)
print(f"\nBest mean accuracy obtained in Cross-Validation (5-fold): {best_accuracy_cv:.4f}")
print(f"\nOptimal hyperparameter combination:")
# Pretty-print the parameters
for param, value in best_params.items():
    print(f"  - {param}: {value}")
print("\nYou can now use these parameters to train your final model.")
print("-" * 50)


🚀 Inizio: Caricamento e preparazione dei dati...
Dati caricati e pronti.
Dimensioni Training Set: (7352, 562)
Dimensioni Test Set: (2947, 562)
--------------------------------------------------
🤖 Avvio dell'ottimizzazione degli iperparametri con Optuna...


Best trial: 5. Best value: 0.979597: 100%|██████████| 15/15 [3:22:58<00:00, 811.88s/it]   


✅ TUNING COMPLETATO

🏆 Miglior accuracy media ottenuta in Cross-Validation (5-fold): 0.9796

⚙️ Combinazione di iperparametri ottimale:
  - n_estimators: 1150
  - learning_rate: 0.27871532239639646
  - max_depth: 2

Ora puoi usare questi parametri per allenare il tuo modello finale.
--------------------------------------------------
